# Resultados · Airbus Ship Segmentation

Este cuaderno **no entrena nada**. Carga los recursos que ya viven en el
proyecto —configuración, pesos, checkpoints— y los usa para reproducir los
resultados que reporta el `README.md`:

| Sección | Qué muestra |
|---|---|
| 1 · Entorno | rutas del repositorio, dispositivo, semilla |
| 2 · Inventario | qué recursos están disponibles y cuáles faltan |
| 3 · Configuración | el YAML que fija el régimen, tal cual se carga |
| 4 · Datos de validación | la partición exacta que la U-Net nunca vio |
| 5 · Modelos | pesos cargados desde `artifacts/` |
| 6 · Métricas | F2, IoU, Dice, precisión, recall y reparto TP/FN/FP |
| 7 · Barrido de umbral | la mejora que no requiere reentrenar |
| 8 · Cómo falla | rejilla cualitativa de predicciones |
| 9 · La cascada | inferencia extremo a extremo, imagen → RLE |
| 10 · Clasificador | accuracy sobre datos retenidos |
| 11 · Checkpoints | lo que quedó registrado del entrenamiento |

Toda la lógica vive en `src/airbus/` y en `scripts/`: aquí solo se importa y se
presenta. Si un número de este cuaderno no coincide con el `README`, la
discrepancia está en los pesos o en los datos, no en dos implementaciones
distintas de la misma métrica.

### Antes de ejecutar

Hacen falta dos cosas que **no se versionan** (ver `.gitignore`):

1. **Los pesos**, en `artifacts/`: `unet.pth`, `classifier_resnet50.pth` y, si
   los hay, los dos `*_checkpoint.pth`. Se producen con
   `scripts/train_segmenter.py` y `scripts/train_classifier.py`, o se descargan
   de donde se hayan publicado.
2. **El dataset** de Kaggle (~30 GB), apuntado desde `configs/default.yaml` o
   desde las constantes de la celda siguiente.

La celda 2 dice exactamente qué falta antes de que nada intente cargarse.

---

## 1 · Entorno y rutas

El cuaderno se ejecuta desde `notebooks/`, así que localiza la raíz del
repositorio subiendo hasta encontrar `pyproject.toml`. Con eso añade `src/` y
`scripts/` al path: no hace falta haber instalado el paquete, aunque
`pip install -e .` también funciona.

In [ ]:
from __future__ import annotations

import sys
from pathlib import Path


def repo_root(start: Path) -> Path:
    """Sube por el árbol hasta el directorio que contiene `pyproject.toml`."""
    for candidate in (start, *start.parents):
        if (candidate / "pyproject.toml").is_file():
            return candidate
    raise RuntimeError(f"No se encontró la raíz del repositorio partiendo de {start}")


ROOT = repo_root(Path.cwd().resolve())

# `src` para importar `airbus`; `scripts` para reutilizar el barrido de umbral
# que ya está escrito en scripts/evaluate.py en lugar de copiarlo aquí.
for extra in (ROOT / "src", ROOT / "scripts"):
    if str(extra) not in sys.path:
        sys.path.insert(0, str(extra))

# ── recursos ───────────────────────────────────────────────────────────────
CONFIG_PATH = ROOT / "configs" / "default.yaml"
ARTIFACTS = ROOT / "artifacts"

PESOS_UNET = ARTIFACTS / "unet.pth"
PESOS_CLASIFICADOR = ARTIFACTS / "classifier_resnet50.pth"
CHECKPOINT_UNET = ARTIFACTS / "unet_checkpoint.pth"
CHECKPOINT_CLASIFICADOR = ARTIFACTS / "classifier_checkpoint.pth"

# Los datos: `None` respeta lo que diga el YAML (rutas de Kaggle). En local,
# apunta aquí a donde hayas descomprimido el dataset.
CSV_LOCAL = None          # p. ej. ROOT / "data" / "train_ship_segmentations_v2.csv"
IMAGENES_LOCAL = None     # p. ej. ROOT / "data" / "train_v2"

# Recorta el conjunto de validación para una pasada rápida. `None` = completo
# (8 512 imágenes de segmentación; en CPU son horas, en GPU minutos).
LIMITE = None

print(f"raíz del repositorio : {ROOT}")
print(f"configuración        : {CONFIG_PATH.relative_to(ROOT)}")
print(f"artefactos           : {ARTIFACTS.relative_to(ROOT)}")

## 2 · Inventario de recursos

Antes de cargar nada, el cuaderno declara qué tiene y qué le falta. Cada sección
de más abajo indica de qué recursos depende, así que un inventario incompleto no
impide ejecutar el resto: solo acota qué se puede reproducir.

In [ ]:
import pandas as pd
import yaml


def _tamano(path: Path) -> str:
    if not path.exists():
        return "—"
    if path.is_dir():
        return f"{sum(1 for _ in path.iterdir()):,} ficheros"
    return f"{path.stat().st_size / 1024**2:,.1f} MB"


_config_bruta = yaml.safe_load(CONFIG_PATH.read_text(encoding="utf-8")) if CONFIG_PATH.is_file() else {}
_rutas = _config_bruta.get("paths", {})
CSV = Path(CSV_LOCAL) if CSV_LOCAL else Path(_rutas.get("csv", ""))
IMAGENES = Path(IMAGENES_LOCAL) if IMAGENES_LOCAL else Path(_rutas.get("train_images", ""))

recursos = [
    ("configuración", CONFIG_PATH, "todas"),
    ("pesos U-Net", PESOS_UNET, "5 · 6 · 7 · 8 · 9"),
    ("pesos ResNet-50", PESOS_CLASIFICADOR, "5 · 9 · 10"),
    ("checkpoint U-Net", CHECKPOINT_UNET, "11"),
    ("checkpoint ResNet-50", CHECKPOINT_CLASIFICADOR, "11"),
    ("CSV de máscaras", CSV, "4 · 6 · 7 · 8 · 9 · 10"),
    ("imágenes de entrenamiento", IMAGENES, "4 · 6 · 7 · 8 · 9 · 10"),
]

inventario = pd.DataFrame(
    [
        {
            "recurso": nombre,
            "disponible": "sí" if ruta.exists() else "NO",
            "tamaño": _tamano(ruta),
            "secciones": secciones,
            "ruta": str(ruta),
        }
        for nombre, ruta, secciones in recursos
    ]
)

faltan = inventario[inventario["disponible"] == "NO"]["recurso"].tolist()
if faltan:
    print(f"⚠ faltan {len(faltan)} recursos: {', '.join(faltan)}")
else:
    print("✔ todos los recursos disponibles")

inventario

In [ ]:
def exigir(*rutas: Path) -> None:
    """Corta con un mensaje claro en lugar de fallar dentro de PyTorch.

    Se llama al principio de cada sección que carga algo de disco: es preferible
    un error explícito aquí que un stack trace de `torch.load` diez líneas más
    abajo.
    """
    ausentes = [str(r) for r in rutas if not Path(r).exists()]
    if ausentes:
        raise FileNotFoundError(
            "No están disponibles estos recursos:\n  - " + "\n  - ".join(ausentes)
            + "\n\nRevisa la sección 1 (constantes de ruta) o entrena con scripts/."
        )

## 3 · Configuración

`configs/default.yaml` reproduce exactamente el régimen del notebook original.
Se carga con `airbus.config.Config`, que es **tipada y estricta**: una clave mal
escrita lanza `TypeError` con la lista de claves válidas, en lugar de ignorarse
en silencio.

In [ ]:
import torch

from airbus.config import Config
from airbus.utils import get_device, set_seed

exigir(CONFIG_PATH)

config = Config.from_yaml(CONFIG_PATH)

# Las rutas locales, si se declararon, ganan a las del YAML: el fichero apunta a
# /kaggle/input y este cuaderno se ejecuta normalmente fuera de Kaggle.
if CSV_LOCAL:
    config.paths.csv = str(CSV_LOCAL)
if IMAGENES_LOCAL:
    config.paths.train_images = str(IMAGENES_LOCAL)

set_seed(config.seed)          # misma semilla → misma partición que el entrenamiento
device = get_device()          # CUDA → MPS → CPU

print(f"dispositivo : {device}")
print(f"semilla     : {config.seed}")

pd.DataFrame(
    [
        {"sección": "segmenter", "clave": k, "valor": str(v)}
        for k, v in config.segmenter.__dict__.items()
    ]
    + [
        {"sección": "classifier", "clave": k, "valor": str(v)}
        for k, v in config.classifier.__dict__.items()
    ]
)

## 4 · Los datos de validación

La partición se reconstruye, no se guarda: `split_dataframe` usa
`random_state=seed`, así que con la misma semilla y el mismo CSV devuelve
exactamente las mismas 8 512 imágenes que la U-Net nunca vio durante el
entrenamiento.

Dos detalles que condicionan todo lo que sigue:

- `segmentation_dataframe` se queda **solo con las imágenes que contienen
  barco** (`EncodedPixels.notnull()`). La U-Net nunca vio océano vacío: por eso
  el sistema necesita el clasificador delante como puerta.
- Las transformaciones se construyen con `train=False`, es decir **sin
  aleatoriedad**. Si la validación fuese estocástica, dos evaluaciones del mismo
  modelo darían números distintos.

In [ ]:
from torch.utils.data import DataLoader

from airbus.data import (
    AirbusSegmentationDataset,
    build_segmentation_transforms,
    load_dataframe,
    segmentation_dataframe,
    split_dataframe,
)

exigir(CSV, IMAGENES)
cfg = config.segmenter

frame = segmentation_dataframe(load_dataframe(config.paths.csv))
if LIMITE:
    frame = frame.head(LIMITE)

train_df, val_df = split_dataframe(frame, cfg.val_fraction, config.seed)

val_dataset = AirbusSegmentationDataset(
    val_df,
    config.paths.train_images,
    build_segmentation_transforms(cfg.image_size, train=False),
)
val_loader = DataLoader(
    val_dataset, batch_size=cfg.batch_size, shuffle=False, num_workers=cfg.num_workers
)

print(f"imágenes con barco : {len(frame):,}")
print(f"train              : {len(train_df):,}")
print(f"validación         : {len(val_df):,}")
val_df.head()

## 5 · Los modelos entrenados

`load_weights` carga con `weights_only=True`: el fichero se deserializa como
tensores puros, sin ejecutar código arbitrario incrustado en el pickle. La
arquitectura se reconstruye desde el YAML (`segmenter.features`), de modo que
los pesos solo encajan si la configuración es la que los produjo.

In [ ]:
from airbus.models import AirbusClassifier, UNet
from airbus.utils import load_weights

exigir(PESOS_UNET, PESOS_CLASIFICADOR)

unet = UNet(in_channels=3, out_channels=1, features=cfg.features)
load_weights(unet, PESOS_UNET, device)
unet.to(device).eval()

clasificador = AirbusClassifier(pretrained=False)
load_weights(clasificador, PESOS_CLASIFICADOR, device)
clasificador.to(device).eval()

pd.DataFrame(
    [
        {
            "modelo": "U-Net (segmentador)",
            "parámetros": f"{sum(p.numel() for p in unet.parameters()):,}",
            "entrada": f"{cfg.image_size}×{cfg.image_size}",
            "pesos": f"{PESOS_UNET.stat().st_size / 1024**2:,.1f} MB",
        },
        {
            "modelo": "ResNet-50 (puerta)",
            "parámetros": f"{sum(p.numel() for p in clasificador.parameters()):,}",
            "entrada": f"{config.classifier.image_size}×{config.classifier.image_size}",
            "pesos": f"{PESOS_CLASIFICADOR.stat().st_size / 1024**2:,.1f} MB",
        },
    ]
)

## 6 · Métricas de segmentación

`evaluate_segmenter` acumula la matriz de confusión **píxel a píxel sobre todo
el conjunto** y calcula las razones al final. No promedia el valor de cada
imagen: eso daría el mismo peso a un carguero de 600 px que a una lancha de 20,
y una sola imagen difícil hundiría la media.

La métrica oficial es **F2**, que pondera el recall cuatro veces más que la
precisión — en vigilancia marítima, omitir un barco es peor que dar una falsa
alarma.

In [ ]:
from airbus.engine import evaluate_segmenter
from airbus.losses import BCEDiceLoss

criterio = BCEDiceLoss(bce_weight=cfg.bce_weight)
metricas, val_loss = evaluate_segmenter(unet, val_loader, device, cfg.threshold, criterio)

print(f"umbral       : {cfg.threshold}")
print(f"pérdida val  : {val_loss:.4f}")

pd.DataFrame(
    [{"métrica": k, "valor": f"{v:.4f}"} for k, v in metricas.as_dict().items()]
)

In [ ]:
# Reparto del área TP + FN + FP. El fondo (TN) se excluye a propósito: son ~99,5 %
# de los píxeles y acertarlos es gratis, así que incluirlos diluiría el reparto
# hasta volverlo ilegible. El porcentaje de TP sobre este total **es el IoU**.
area = metricas.tp + metricas.fn + metricas.fp

reparto = pd.DataFrame(
    [
        ("TP · acertados", metricas.tp),
        ("FN · omitidos", metricas.fn),
        ("FP · falsa alarma", metricas.fp),
    ],
    columns=["píxeles de", "cantidad"],
)
reparto["%"] = (reparto["cantidad"] / area * 100).round(2)
reparto["cantidad"] = reparto["cantidad"].map("{:,}".format)

sesgo = "conservadora" if metricas.precision > metricas.recall else "agresiva"
print(f"total TP+FN+FP : {area:,} píxeles")
print(f"la red es {sesgo}: precisión {metricas.precision:.4f} vs recall {metricas.recall:.4f}")
reparto

## 7 · Barrido de umbral

0,5 es el valor **neutro**, no el óptimo. Si la red es conservadora
(precisión > recall) y la métrica pondera el recall ×4, bajar el umbral sube la
puntuación sin tocar un solo peso: es la mejora más barata disponible.

La función es la misma que usa `scripts/evaluate.py --sweep`, importada de ahí.
Acumula las probabilidades una vez y las umbraliza N veces, en lugar de releer
el dataset por cada umbral.

In [ ]:
import numpy as np

from evaluate import sweep_threshold   # scripts/evaluate.py

umbrales = np.round(np.arange(0.1, 0.95, 0.1), 2)
resultados = sorted(sweep_threshold(unet, val_loader, device, umbrales))

barrido = pd.DataFrame(resultados, columns=["umbral", "F2", "recall"])
mejor = barrido.loc[barrido["F2"].idxmax()]

print(f"mejor F2 = {mejor['F2']:.4f} con umbral {mejor['umbral']:.2f}")
print(f"por defecto (0,50) = {metricas.f2:.4f}  →  ganancia {mejor['F2'] - metricas.f2:+.4f}")
barrido.round(4)

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(8, 4.5))
ax.plot(barrido["umbral"], barrido["F2"], marker="o", label="F2 (oficial)")
ax.plot(barrido["umbral"], barrido["recall"], marker="s", label="recall")
ax.axvline(cfg.threshold, ls="--", lw=1, color="grey", label=f"umbral por defecto ({cfg.threshold})")
ax.scatter([mejor["umbral"]], [mejor["F2"]], s=140, facecolors="none", edgecolors="crimson", zorder=5)
ax.annotate(
    f"máx F2 = {mejor['F2']:.4f}",
    xy=(mejor["umbral"], mejor["F2"]),
    xytext=(6, 10),
    textcoords="offset points",
    color="crimson",
)
ax.set_xlabel("umbral sobre la probabilidad por píxel")
ax.set_ylabel("puntuación")
ax.set_title("Barrido del umbral de decisión · conjunto de validación")
ax.legend()
ax.grid(alpha=0.3)
fig.tight_layout()
plt.show()

## 8 · Cómo falla · rejilla cualitativa

`plot_segmentation_batch` dibuja, por muestra: imagen, máscara real, predicción
y superposición. Los errores no son aleatorios — se concentran en el **contorno**
del casco y en la **estela**, casi nunca en el fondo lejano.

El `DataLoader` de esta sección sí baraja, para no mirar siempre el mismo lote;
la semilla global de la sección 3 lo mantiene reproducible.

In [ ]:
from airbus.utils.viz import plot_segmentation_batch

muestra_loader = DataLoader(val_dataset, batch_size=8, shuffle=True)

fig = plot_segmentation_batch(
    unet, muestra_loader, device, num_samples=4, threshold=cfg.threshold
)
plt.show()

# Para guardar la rejilla en disco, pasa `save_path`:
# plot_segmentation_batch(..., save_path=ROOT / "figures" / "predicciones.png")

## 9 · La cascada extremo a extremo

Lo que el notebook original no llegaba a tener: las dos redes encadenadas.
`ShipSegmentationPipeline` recibe una imagen PIL y devuelve la máscara a
resolución nativa (768 × 768) más el RLE listo para enviar a Kaggle.

Dos decisiones dentro del pipeline que se ven en los números de abajo:

- Si la puerta dice «no hay barco», **la U-Net ni se ejecuta**: máscara vacía y
  salida temprana. El 77,9 % del dataset sale por esa rama barata.
- Los logits se reescalan a 768 × 768 y **después** se umbralizan: interpolar
  una máscara ya binarizada introduce escalones en el contorno.

In [ ]:
from PIL import Image

from airbus.data import rle_decode
from airbus.pipeline import ShipSegmentationPipeline

N_MUESTRAS = 8

pipeline = ShipSegmentationPipeline(
    clasificador,
    unet,
    device,
    classifier_size=config.classifier.image_size,
    segmenter_size=cfg.image_size,
    mask_threshold=cfg.threshold,
)

filas = []
for _, fila in val_df.head(N_MUESTRAS).iterrows():
    ruta = Path(config.paths.train_images) / fila["ImageId"]
    prediccion = pipeline.predict(Image.open(ruta))
    rle = prediccion.to_rle()
    filas.append(
        {
            "ImageId": fila["ImageId"],
            "p(barco)": round(prediccion.ship_probability, 4),
            "puerta": "pasa" if prediccion.has_ship else "corta",
            "píxeles predichos": int(prediccion.mask.sum()),
            "píxeles reales": int(rle_decode(fila["EncodedPixels"]).sum()),
            "RLE": (rle[:36] + "…") if len(rle) > 36 else (rle or "(vacío)"),
        }
    )

cascada = pd.DataFrame(filas)
cortadas = int((cascada["puerta"] == "corta").sum())
print(f"todas estas imágenes contienen barco; la puerta cortó {cortadas} de {len(cascada)}")
cascada

## 10 · El clasificador sobre datos retenidos

El notebook original creaba la partición de validación y **nunca la usaba**, así
que su accuracy de 0,9429 era de entrenamiento. `validate_classifier` es lo que
faltaba para distinguir aprendizaje de memorización.

La partición se rehace igual que en `scripts/train_classifier.py`: submuestreo
50/50 (`negative_ratio`) y `stratify_on='has_ship'`, para que el equilibrio
sobreviva al corte. Con el reparto equilibrado, el baseline trivial de responder
siempre «no hay barco» vale 0,50 — no 0,779.

> Es la sección más cara del cuaderno: recorre ~13 600 imágenes a 224 × 224.
> Sube `LIMITE` en la sección 1 para una pasada corta.

In [ ]:
import torch.nn as nn

from airbus.data import (
    AirbusClassificationDataset,
    balance_by_undersampling,
    build_classifier_transforms,
    image_level_dataframe,
)
from airbus.engine import validate_classifier

cfg_clf = config.classifier

imagenes = image_level_dataframe(load_dataframe(config.paths.csv))
balanceado = balance_by_undersampling(imagenes, seed=config.seed, ratio=cfg_clf.negative_ratio)
if LIMITE:
    balanceado = balanceado.head(LIMITE)

_, val_clf_df = split_dataframe(
    balanceado, cfg_clf.val_fraction, config.seed, stratify_on="has_ship"
)

val_clf_loader = DataLoader(
    AirbusClassificationDataset(
        val_clf_df,
        config.paths.train_images,
        build_classifier_transforms(cfg_clf.image_size, train=False),
    ),
    batch_size=cfg_clf.batch_size,
    shuffle=False,
    num_workers=cfg_clf.num_workers,
)

positivos = int(val_clf_df["has_ship"].sum())
print(f"validación : {len(val_clf_df):,} imágenes")
print(f"reparto    : {positivos:,} con barco / {len(val_clf_df) - positivos:,} sin barco")

resultado = validate_classifier(clasificador, val_clf_loader, nn.BCEWithLogitsLoss(), device)
print(f"\naccuracy de validación : {resultado.accuracy:.4f}")
print(f"pérdida de validación  : {resultado.loss:.4f}")
print(f"baseline trivial       : {max(positivos, len(val_clf_df) - positivos) / len(val_clf_df):.4f}")

## 11 · Qué quedó registrado en los checkpoints

Pesos y checkpoint son dos artefactos distintos: el `state_dict` es lo mínimo
para hacer inferencia; el checkpoint añade **el estado del optimizador**, la
época y las métricas del momento en que se guardó.

Aquí solo se leen los campos escalares, sin reconstruir ningún modelo: basta
para saber en qué época se quedó el mejor resultado y con qué métricas.

In [ ]:
def leer_extras(path: Path) -> dict:
    """Campos no-tensoriales de un checkpoint (época, pérdida, métricas)."""
    checkpoint = torch.load(path, map_location="cpu", weights_only=True)
    return {k: v for k, v in checkpoint.items() if not k.endswith("state_dict")}


for etiqueta, path in [
    ("U-Net", CHECKPOINT_UNET),
    ("ResNet-50", CHECKPOINT_CLASIFICADOR),
]:
    print(f"── {etiqueta} " + "─" * (60 - len(etiqueta)))
    if not path.exists():
        print(f"   (no disponible: {path})\n")
        continue
    for clave, valor in leer_extras(path).items():
        if isinstance(valor, dict):
            valor = "  ".join(f"{k}={v:.4f}" for k, v in valor.items())
        elif isinstance(valor, float):
            valor = f"{valor:.4f}"
        print(f"   {clave:<12} {valor}")
    print()

In [ ]:
# El factor 3 del checkpoint es la forma más rápida de confirmar que el estado
# del optimizador se guardó de verdad: Adam mantiene dos momentos por parámetro,
# así que pesos + momento 1 + momento 2 ≈ 3× el tamaño del modelo.
if PESOS_UNET.exists() and CHECKPOINT_UNET.exists():
    pesos_mb = PESOS_UNET.stat().st_size / 1024**2
    ckpt_mb = CHECKPOINT_UNET.stat().st_size / 1024**2
    print(f"unet.pth            : {pesos_mb:7,.1f} MB")
    print(f"unet_checkpoint.pth : {ckpt_mb:7,.1f} MB")
    print(f"razón               : {ckpt_mb / pesos_mb:7.2f}×  (esperado ≈ 3)")
else:
    print("faltan pesos o checkpoint de la U-Net; nada que comparar.")

---

## Y a partir de aquí

Lo que este cuaderno deja preparado y no ejecuta:

- **Enviar a Kaggle** — `scripts/predict.py` recorre una carpeta con el mismo
  pipeline de la sección 9 y escribe el CSV de envío.
- **Fijar el umbral óptimo** — si la sección 7 encontró un umbral mejor que 0,5,
  cámbialo en `configs/default.yaml` (`segmenter.threshold`) y todo el proyecto
  lo hereda: no hay ningún 0,5 escrito a mano en el código.
- **Comparar regímenes** — copia el YAML, cambia un valor
  (`classifier.freeze_backbone`, `segmenter.bce_weight`, `features`…) y pásalo
  con `--config`. Este cuaderno apunta al nuevo fichero cambiando `CONFIG_PATH`
  en la sección 1.

El detalle de por qué cada pieza es como es está en el `README.md`; el registro
del desarrollo original, en `notebooks/original/Airbus_Segmentation_v3.ipynb`.